## Logistic Regression + TF-IDF Vectorization

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    classification_report
)
from sklearn.model_selection import train_test_split
import seaborn as sns

In [ ]:
dataset = load_dataset(
    "json",
    data_files="/content/datasets/*.json"
)

dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [ ]:
train_df = pd.DataFrame(train_dataset)
test_df = pd.DataFrame(test_dataset)

print(train_df.head())

In [ ]:

X_train = train_df["input"]
y_train = train_df["label"].astype(int)

X_test = test_df["input"]
y_test = test_df["label"].astype(int)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),   # unigrams + bigrams
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

print("ROC-AUC:", auc_score)

In [ ]:
custom_threshold = 0.7

custom_preds = (y_prob >= custom_threshold).astype(int)

print("Custom Threshold:", custom_threshold)
print("Accuracy:", accuracy_score(y_test, custom_preds))
print("F1 Score:", f1_score(y_test, custom_preds))

In [ ]:
import joblib

joblib.dump(model, "logistic_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

## Testing with pdf


In [ ]:
import joblib
import numpy as np

model = joblib.load("logistic_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

In [ ]:
def score_sentences(sentences):
    scores = []

    # Convert sentences → TF-IDF
    sentence_vectors = vectorizer.transform(sentences)

    # Get probabilities
    probs = model.predict_proba(sentence_vectors)[:, 1]

    for sentence, prob in zip(sentences, probs):
        scores.append((sentence, prob))

    return scores

In [ ]:
!pip install pdfplumber scikit-learn -q

In [ ]:
import re
import pdfplumber
from pathlib import Path

def extract_text_from_pdf(pdf_path):
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                all_text += page_text + "\n"

    # Remove URLs
    all_text = re.sub(r"http\S+", "", all_text)
    all_text = re.sub(r"www\.\S+", "", all_text)

    return all_text

In [ ]:
def remove_consecutive_duplicates(text):
    lines = text.split("\n")
    cleaned = []
    prev_line = None

    for line in lines:
        stripped = line.strip()
        if stripped == prev_line:
            continue
        cleaned.append(line)
        prev_line = stripped

    return "\n".join(cleaned)


def remove_equation_lines(text):
    cleaned_lines = []

    for line in text.split("\n"):
        stripped = line.strip()

        if not stripped:
            cleaned_lines.append("")
            continue

        math_symbol_ratio = sum(c in "𝜃𝑥𝑦𝑋𝑌𝒚𝜽=∗+-^()⋯" for c in stripped) / len(stripped)

        if math_symbol_ratio > 0.3:
            continue

        if re.fullmatch(r"[0-9\s=+\-*/().]+", stripped):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def clean_text_structured(text: str) -> str:
    text = text.replace('\ufeff', '')
    text = re.sub(r'\r\n', '\n', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[•◦▪–—]', '-', text)

    lines = [l.strip() for l in text.split('\n')]

    cleaned = []
    buffer = ""

    for line in lines:
        if not line:
            if buffer:
                cleaned.append(buffer)
                buffer = ""
            cleaned.append("")
            continue

        if line.startswith("-"):
            if buffer:
                cleaned.append(buffer)
                buffer = ""
            cleaned.append(line)
            continue

        if buffer and not re.search(r'[.!?]$', buffer):
            buffer += " " + line
        else:
            if buffer:
                cleaned.append(buffer)
            buffer = line

    if buffer:
        cleaned.append(buffer)

    text = "\n".join(cleaned)
    text = re.sub(r'\n{3,}', '\n\n', text)

    text = remove_consecutive_duplicates(text)
    text = remove_equation_lines(text)

    return text.strip()

In [ ]:
def split_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return sentences

In [ ]:
def select_top_sentences(scored_sentences, ratio=0.25):
    sorted_by_score = sorted(scored_sentences, key=lambda x: x[1], reverse=True)

    top_k = max(1, int(len(sorted_by_score) * ratio))
    selected = sorted_by_score[:top_k]

    selected_sentences = [s[0] for s in selected]

    original_order = [s[0] for s in scored_sentences]
    selected_sentences.sort(key=lambda x: original_order.index(x))

    return selected_sentences

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def remove_redundancy(sentences, threshold=0.8):
    if len(sentences) <= 1:
        return sentences

    vectorizer = TfidfVectorizer().fit_transform(sentences)
    similarity_matrix = cosine_similarity(vectorizer)

    filtered = []
    used = set()

    for i in range(len(sentences)):
        if i in used:
            continue

        filtered.append(sentences[i])

        for j in range(i+1, len(sentences)):
            if similarity_matrix[i][j] > threshold:
                used.add(j)

    return filtered

In [ ]:
def format_as_bullets(sentences):
    return "\n".join([f"- {s}" for s in sentences])

In [ ]:
def summarize_lecture(pdf_path, ratio=0.3):

    # 1️⃣ Extract
    raw_text = extract_text_from_pdf(pdf_path)

    # 2️⃣ Clean
    cleaned_text = clean_text_structured(raw_text)

    # 3️⃣ Split
    sentences = split_into_sentences(cleaned_text)

    # 4️⃣ Score with transformer
    scored = score_sentences(sentences)

    # 5️⃣ Select top sentences
    selected = select_top_sentences(scored, ratio)

    # 6️⃣ Remove redundancy
    final_sentences = remove_redundancy(selected)

    # 7️⃣ Bullet formatting
    summary = format_as_bullets(final_sentences)

    return summary

In [ ]:
import re

def clean_bullet_text(text):
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Remove duplicated bullet symbols like "- -", "--", "• -"
        line = re.sub(r'^[-•\s]+', '', line)

        # Skip empty lines
        if not line:
            continue

        # Add single clean bullet
        cleaned_lines.append(f"- {line}")

    return "\n".join(cleaned_lines)

In [ ]:
test_data_path = "/content/test/Artificial Intelligence Technology_lecture 7.pdf"
summary = clean_bullet_text(summarize_lecture(test_data_path, ratio=0.3))
# summary = summarize_lecture(test_data_path, ratio=0.3)
print(summary)

In [ ]:
test_data_path = "/content/test/Artificial Intelligence Technology_lecture 9.pdf"
summary = clean_bullet_text(summarize_lecture(test_data_path, ratio=0.3))
print(summary)